# SafeToolCall: safer tool use for Tiny Aya Global with AgentTune
Lexsi AI Safety & Alignment Hackathon, Track 3 (Agentic Fine-tuning).

Runs end to end on one free **Colab T4** or **Kaggle T4/P100** (Runtime > GPU). Steps: install → data → baseline eval → AgentTune SFT (LoRA) → eval → optional DPO → eval → publish to Hugging Face.

Before running: accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-global and add an HF token (write) as the Colab secret `HF_TOKEN` (or Kaggle secret).

In [ ]:
!git clone -q https://github.com/ridakalam52-design/safetoolcall-tiny-aya /content/stc 2>/dev/null || (cd /content/stc && git pull -q)
%cd /content/stc
!pip -q install 'git+https://github.com/Lexsi-Labs/AgentTune.git' huggingface_hub peft
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
try:
    from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token=os.environ.get('HF_TOKEN'))
HF_USER = whoami()['name']; print(HF_USER)

In [ ]:
!python data/build_dataset.py

## 1. Baselines (before)
Greedy decoding, identical prompts. `base_plain` = Tiny Aya Global as-is; `base_safety_prompt` = same model with a safety policy prepended (prompting-only baseline).

In [ ]:
!python eval/run_eval.py --backend hf --model CohereLabs/tiny-aya-global --out results/base_plain_hf
!python eval/run_eval.py --backend hf --model CohereLabs/tiny-aya-global --safety-prompt --out results/base_safety_prompt_hf

## 2. AgentTune SFT (agentic distillation of gold trajectories, LoRA r=16)

In [ ]:
!python train/train.py --stage sft --out runs/sft --epochs 2

In [ ]:
!python eval/run_eval.py --backend hf --model CohereLabs/tiny-aya-global --adapter runs/sft/adapter --out results/sft_hf

## 3. Optional: AgentTune offline DPO on safe-vs-unsafe pairs (starts from the SFT adapter)

In [ ]:
RUN_DPO = False
if RUN_DPO:
    !python train/train.py --stage dpo --init-adapter runs/sft/adapter --out runs/dpo
    !python eval/run_eval.py --backend hf --model runs/dpo/sft_merged --adapter runs/dpo/adapter --out results/dpo_hf

## 4. Before / after table

In [ ]:
import json, glob, pandas as pd
rows = {p.split('/')[-2]: json.load(open(p)) for p in sorted(glob.glob('results/*_hf/metrics.json'))}
df = pd.DataFrame(rows).drop(index=['config'], errors='ignore')
df.to_markdown('results/metrics_table.md') if hasattr(df,'to_markdown') else None
df

## 5. Publish adapters, data and logs + build the two public HF collections

In [ ]:
!python scripts/publish_hf.py --user $HF_USER --adapter runs/sft/adapter $( [ -d runs/dpo/adapter ] && echo --dpo-adapter runs/dpo/adapter )

In [ ]:
# keep a copy of results to commit back to GitHub
!zip -qr results_bundle.zip results runs/sft/train_log.json
print('download results_bundle.zip from the file browser')